**Задание 9.5. Модуль ML-6 (HW-03)**

Обучите модель линейной регрессии на найденных двумя способами трёх важных признаках и сравните полученные результаты.

КРИТЕРИИ ОЦЕНИВАНИЯ:

1.	Верно выделены три столбца-признака для обучения, выбранные RFE.
2.	Верно выделены три столбца-признака для обучения, выбранные SelectKBest.
3.	Обучена регрессия на первых трёх столбцах, оценено качество модели на тесте.
4.	Обучена регрессия на вторых трёх столбцах, оценено качество модели на тесте.
5.	Произведено сравнение выбранных метрик в форме комментария. Дан ответ на вопрос «Какой метод отбора признаков показал наилучший результат на тестовой выборке?» (в текстовой ячейке).

In [37]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn import metrics #метрики

In [38]:
data = pd.read_excel('data/data_ford_price.xlsx') 

## Предобработка данных

In [39]:
data = data[['price','year', 'cylinders', 'odometer', 'lat', 'long', 'weather']]
data.dropna(inplace = True)

y = data['price']
x = data.drop(columns='price')

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=40)



### RFE

In [40]:
from sklearn.feature_selection import RFE

In [41]:
# Выбор столбцов-признаков для обучения
estimator = LinearRegression()
selector = RFE(estimator, n_features_to_select=3, step=1)
selector = selector.fit(X_train, y_train)
 
rfe_columns=selector.get_feature_names_out()
rfe_columns

array(['year', 'cylinders', 'lat'], dtype=object)

In [42]:
# Обучение модели

X_train, X_test, y_train, y_test = train_test_split(x[rfe_columns], y, test_size=0.3, random_state=40)
model = LinearRegression()

model.fit(X_train, y_train)
y_predicted = model.predict(X_test)
 
mae = mean_absolute_error(y_test, y_predicted)
print('MAE: %.3f' % mae)

MAE: 5096.570


###  SelectKBest

In [43]:
from sklearn.feature_selection import SelectKBest, f_classif

In [44]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=40)

In [45]:
# Выбор столбцов-признаков для обучения
selector = SelectKBest(f_classif, k=3)
selector.fit(X_train, y_train)
 
skb_columns=selector.get_feature_names_out()
skb_columns

array(['year', 'cylinders', 'odometer'], dtype=object)

In [46]:
# Обучение модели
X_train, X_test, y_train, y_test = train_test_split(x[skb_columns], y, test_size=0.3, random_state=40)
model = LinearRegression()
model.fit(X_train, y_train)
y_predicted = model.predict(X_test)
 
mae = mean_absolute_error(y_test, y_predicted)
print('MAE: %.3f' % mae)

MAE: 4708.946


### Вывод

На тестовой выборке наилучший результат показал метод отбора SelectKBest, т.к. в этом случае средняя абсолютная ошибка (MAE=4708.946) меньше, чем средняя абсолютная ошибка (MAE=5096.570), полученная при применении метода RFE.